In [23]:
!pip install -q langchain langchain-core langchain-community langchain-huggingface langchain-chroma langchain-google-genai chromadb pypdf sentence-transformers gradio

In [24]:
import os
from getpass import getpass
from google import genai
from google.colab import userdata


#if "GOOGLE_API_KEY" not in os.environ:
  #  os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google Gemini API Key: ")


#API_KEY = put the new api key here created latest
api_key = userdata.get("GEMINI_KEY")
#client = genai.Client(api_key=api_key)
os.environ["GEMINI_KEY"] = api_key

In [25]:
import gradio as gr
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

# Initialize Embedding Model & Vectorstore placeholder
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = None
retriever = None

# Initialize LLM
llm = ChatGoogleGenerativeAI(model="gemini-1.0-pro", temperature=0.3, api_key=os.environ["GEMINI_KEY"])

# Global tracker for user mastery
mastery_log = []

def process_document(pdf_file):
    """Processes uploaded PDF and indexes it into ChromaDB vector database."""
    global vectorstore, retriever
    if not pdf_file:
        return "Please upload a PDF document first."

    # Load and split PDF
    loader = PyPDFLoader(pdf_file.name)
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    splits = text_splitter.split_documents(docs)

    # Build Vector Index
    vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    return f"✅ Successfully processed '{os.path.basename(pdf_file.name)}'! Loaded {len(splits)} knowledge chunks."

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [26]:
def explain_concept(concept, style):
    """Explains difficult concepts using context from uploaded material in varied styles."""
    if not retriever:
        return "⚠️ Please upload study material first!"

    docs = retriever.invoke(concept)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
    You are an expert AI Tutor named Smart Study Buddy.
    Explain the concept of '{concept}' based on the following material:

    Context:
    {context}

    Explanation Style: {style}
    (Options: 'Simple 5-year-old metaphor', 'Detailed academic breakdown', 'Practical real-world example')

    Keep the answer concise, engaging, and structured with bullet points where useful.
    """

    response = llm.invoke(prompt)
    return response.content


def generate_quiz(topic, num_questions):
    """Generates custom practice questions based on indexed material."""
    if not retriever:
        return "⚠️ Please upload study material first!"

    docs = retriever.invoke(topic)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
    Create a {num_questions}-question practice quiz on the topic '{topic}' using the context below.
    Include multiple-choice options (A, B, C, D) and hide the correct answers with explanations at the bottom under an 'Answer Key' section.

    Context:
    {context}
    """

    response = llm.invoke(prompt)
    return response.content


def track_progress(topic, score_status):
    """Tracks concepts mastered or requiring review."""
    global mastery_log
    status_icon = "✅ Mastered" if score_status == "Passed" else "⚠️ Needs Review"
    mastery_log.append({"Topic": topic, "Status": status_icon})

    report = "### 📊 Learning Progress Dashboard\n"
    for idx, entry in enumerate(mastery_log, 1):
        report += f"{idx}. **{entry['Topic']}** — {entry['Status']}\n"
    return report


def generate_schedule(available_hours, target_days):
    """Generates a personalized study schedule based on user inputs."""
    prompt = f"""
    Create a structured, step-by-step {target_days}-day study plan for a student who has {available_hours} hours available per day.
    Divide the time logically into active recall, concept review, practice quizzes, and break periods.
    Output the plan in a clear Markdown table.
    """
    response = llm.invoke(prompt)
    return response.content

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="Smart Study Buddy 🎓") as app:
    gr.Markdown("# 🎓 Smart Study Buddy — AI RAG Learning Assistant")
    gr.Markdown("Upload your study material (PDF) and let your AI tutor help you master concepts, test your knowledge, and plan your study routine.")

    with gr.Row():
        pdf_input = gr.File(label="Upload Textbook / Study Material (PDF)", file_types=[".pdf"])
        upload_btn = gr.Button("⚡ Index Document", variant="primary")

    upload_status = gr.Textbox(label="System Status", interactive=False)
    upload_btn.click(process_document, inputs=[pdf_input], outputs=[upload_status])

    with gr.Tabs():
        # Tab 1: Concept Explainer
        with gr.TabItem("💡 Concept Explainer"):
            concept_input = gr.Textbox(label="What concept do you want explained?", placeholder="e.g., Photosynthesis, Neural Networks, Supply and Demand")
            style_input = gr.Radio(
                ["Simple 5-year-old metaphor", "Detailed academic breakdown", "Practical real-world example"],
                label="Explanation Style",
                value="Simple 5-year-old metaphor"
            )
            explain_btn = gr.Button("Explain Concept", variant="primary")
            explain_output = gr.Markdown(label="Explanation Output")
            explain_btn.click(explain_concept, inputs=[concept_input, style_input], outputs=[explain_output])

        # Tab 2: Quiz Generator
        with gr.TabItem("📝 Practice Quiz Generator"):
            quiz_topic = gr.Textbox(label="Quiz Topic", placeholder="e.g., Chapter 3, Mitosis, Key Terms")
            num_q = gr.Slider(minimum=1, maximum=10, value=3, step=1, label="Number of Questions")
            quiz_btn = gr.Button("Generate Practice Quiz", variant="primary")
            quiz_output = gr.Markdown(label="Quiz")
            quiz_btn.click(generate_quiz, inputs=[quiz_topic, num_q], outputs=[quiz_output])

        # Tab 3: Mastery Tracker
        with gr.TabItem("📈 Progress & Mastery Tracker"):
            tracker_topic = gr.Textbox(label="Topic Name")
            status_input = gr.Radio(["Passed", "Needs Work"], label="Assessment Status", value="Passed")
            track_btn = gr.Button("Log Progress")
            tracker_output = gr.Markdown(label="Dashboard")
            track_btn.click(track_progress, inputs=[tracker_topic, status_input], outputs=[tracker_output])

        # Tab 4: Study Planner
        with gr.TabItem("📅 Personalized Study Schedule"):
            hours = gr.Number(label="Available Study Hours Per Day", value=2)
            days = gr.Number(label="Number of Days Until Exam/Target", value=7)
            schedule_btn = gr.Button("Generate Schedule", variant="primary")
            schedule_output = gr.Markdown(label="Study Schedule")
            schedule_btn.click(generate_schedule, inputs=[hours, days], outputs=[schedule_output])

# Launch the app in Colab
app.launch(debug=True, share=True)

/tmp/ipykernel_718/2609738247.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Smart Study Buddy 🎓") as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e59eae73f36b74288a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py", line 4177, in _generate
    response: GenerateContentResponse = self.client.models.generate_content(
                                        ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        **request,
        ^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/google/genai/models.py", line 6270, in generate_content
    response = self._generate_content(
        model=model, contents=contents, config=parsed_config_to_call
    )
  File "/usr/local/lib/python3.13/dist-packages/google/genai/models.py", line 4707, in _generate_content
    response = self._api_client.request(
        'post', path, request_dict, http_options
    )
  File "/usr/local/lib/python3.13/dist-packages/google/genai/_api_client.py", line 1750, in request
    response = self._request(http_request, http_options, stream=False)
  File "/usr/local/lib/python3.13/dist-packages/google/gena